In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class CausalSelfAttention(nn.Module):
  """
  i/p: (batch_size, seq_len, embed_dim)
  o/p: (batch_size, seq_len, head_dim)
  """

  def __init__(self, embed_dim:int, max_seq_len:int, head_dim:int):
    super().__init__()
    self.embed_dim = embed_dim
    self.head_dim = head_dim

    # projections: (b, t, e) -> (b, t, h)
    self.query_proj = nn.Linear(embed_dim, head_dim, bias=False)
    self.key_proj = nn.Linear(embed_dim, head_dim, bias=False)
    self.value_proj = nn.Linear(embed_dim, head_dim, bias=False)

    # causal_mask (seq_len, seq_len)
    self.register_buffer(
        "causal_mask",
        torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
    )

  def forward(self, x):
    """
    x : (b, t, e)
    """
    batch_size, seq_len, _ = x.shape

    # q, k, v: (b, t, h)
    W_q = self.query_proj(x)
    W_k = self.key_proj(x)
    W_v = self.value_proj(x)

    attn_logits = W_q @ W_k.transpose(-2, -1)  # (b, t, t)
    attn_logits = attn_logits / math.sqrt(self.head_dim)

    mask = self.causal_mask[:seq_len, :seq_len]
    attn_logits = attn_logits.masked_fill(~mask, float("-inf"))  # (b, t, t)

    attn_weights = F.softmax(attn_logits, dim=-1) # (b, t, t)
    out = attn_weights @ W_v
    return out


class MultiHeadAttention(nn.Module):

  """
  i/p : (b, t, e)
  o/p : (b, t, e)
  """

  def __init__(self, embed_dim:int, max_seq_len:int, num_heads:int):
    super().__init__()
    assert embed_dim % num_heads == 0

    self.embed_dim = embed_dim
    self.num_heads = num_heads
    self.head_dim = embed_dim // num_heads

    # each head returns (b, t, head_dim)
    self.heads = nn.ModuleList(
        [
            CausalSelfAttention(embed_dim, max_seq_len, self.head_dim)
            for _ in range(num_heads)
        ]
    )

    # concate heads
    self.output_proj = nn.Linear(embed_dim, embed_dim)

  def forward(self, x):
    """
    x : (b, t, e)
    """
    head_outputs = [head(x) for head in self.heads]

    # (b, t, h) * num_heads -> (b, t, num_heads*h) = (b, t, e)
    concat = torch.cat(head_outputs, dim=-1)

    # project back to embed_dim
    out = self.output_proj(concat)
    return out


class FeedForward(nn.Module):
  """
  i/p : (b, t, e)
  o/p : (b, t, e)
  """

  def __init__(self, embed_dim:int, expanison:int = 4):
    super().__init__()
    hidden_dim = expanison * embed_dim

    self.layers = nn.Sequential(
        nn.Linear(embed_dim, hidden_dim),  # (b, t, e) -> (b, t, 4e)
        nn.GELU(),
        nn.Linear(hidden_dim, embed_dim),  # (b, t, 4e) -> (b, t, e)
    )

  def forward(self, x):
    return self.layers(x)  # (b, t, e)


class TransformerBlock(nn.Module):
  """
  i/p : (b, t, e)
  o/p : (b, t, e)
  """

  def __init__(self, embed_dim:int, max_seq_len:int, num_heads:int):
    super().__init__()
    self.attn = MultiHeadAttention(embed_dim, max_seq_len, num_heads)
    self.ffn = FeedForward(embed_dim)

    self.attn_norm = nn.LayerNorm(embed_dim)
    self.ffn_norm = nn.LayerNorm(embed_dim)

  def forward(self, x):
    """
    x : (b, t, e)
    """
    x =  x + self.attn(self.attn_norm(x))
    x =  x + self.ffn(self.ffn_norm(x))
    return x

In [3]:
class Tokenizer:

  def __init__(self, special_tokens=None):
    if special_tokens is None:
      special_tokens = ["<PAD>", "<UNK>", "<END>"]
    self.special_tokens = special_tokens

    self.word_to_idx = {}
    self.idx_to_word = {}
    self.vocab_size = 0

    self.pad_token = "<PAD>"
    self.unk_token = "<UNK>"
    self.end_token = "<END>"

    self.pad_id = None
    self.unk_id = None
    self.end_id = None

  def build_vocab(self, texts):

    tokens = []
    for s in texts:
      tokens.extend(self.tokenize(s))

    unique_tokens = sorted(set(tokens))

    vocab = list(self.special_tokens)
    for t in unique_tokens:
      if t not in vocab:
        vocab.append(t)

    self.word_to_idx = {w : i for i, w in enumerate(vocab)}
    self.idx_to_word = {i : w for w, i in self.word_to_idx.items()}
    self.vocab_size = len(vocab)

    # Cache special ids
    self.pad_id = self.word_to_idx[self.pad_token]
    self.unk_id = self.word_to_idx[self.unk_token]
    self.end_id = self.word_to_idx[self.end_token]

    return self

  def tokenize(self, text: str):
    text = text.replace("\n", " ")
    return text.strip().split()

  def encode(self, text, add_end=False):

    tokens = self.tokenize(text)
    if add_end:
      tokens = tokens + [self.end_token]
    return [self.word_to_idx.get(t, self.unk_id) for t in tokens]

  def decode(self, ids, stop_at_end=False):

    words = []
    for i in ids:
      w = self.idx_to_word.get(int(i), self.unk_token)
      if stop_at_end and w == self.end_token:
        break

      if w != self.pad_token:
        words.append(w)

    return " ".join(words)

In [27]:
corpus = [
    "today i built a vision language model completely from scratch",
    "i implemented a vision transformer encoder and a llama style decoder",
    "training the model on a small dataset helped me understand every tensor operation",
    "working with attention mechanisms deepened my understanding of representation learning",
    "handling tensor dimensions is challenging but extremely rewarding",
    "i believe building models from scratch gives clarity beyond theory",
    "recently i designed a multimodal rag system using vector databases",
    "the system indexes both text and images for retrieval based generation",
    "i integrated scalable fastapi backends for production ready ai pipelines",
    "i prefer building frontend applications using typescript and react",
    "i am currently exploring multi agent architectures for reasoning tasks",
    "my focus is on designing reliable and efficient ai systems",
    "i am particularly interested in explainable artificial intelligence",
    "understanding why a model predicts something is as important as accuracy",
    "i am researching supervised classification using retrieval based approaches",
    "my goal is to bridge theoretical machine learning and practical deployment",
    "i enjoy participating in hackathons to test real world problem solving skills",
    "recently i worked on artificial sight and artificial sound systems",
    "the vision is to build technology where the deaf can hear and the blind can see",
    "i am building vidya mitra to provide ai powered learning for rural students",
    "education accessibility through intelligent agents is a long term mission",
    "i am exploring margin based contrastive learning for robust classification",
    "designing scalable ai products requires both research depth and engineering discipline",
    "i strongly believe learning by building accelerates understanding",
    "every project i build improves my systems thinking ability",
    "my work combines computer vision natural language processing and reasoning systems",
    "i aim to publish research that is both rigorous and practically useful",
    "continuous experimentation helps me refine architectural decisions",
    "from retrieval systems to multimodal models i enjoy exploring complex ai pipelines",
    "my objective is to build impactful artificial intelligence systems responsibly"
]

# adding "<END>" token for each sentence
corpus = [s + "<END>" for s in corpus]

tokenizer = Tokenizer().build_vocab(corpus)

encoded_stream = []
for s in corpus:
    encoded_stream.extend(tokenizer.encode(s))
data = torch.tensor(encoded_stream, dtype=torch.long)

print("data:", data)
print("data length:", len(data))

# Hyperparameters
context_len = 6
embedding_dim = 64
n_heads = 2
n_layers = 2
lr = 1e-3
epochs = 1500

def get_batch(batch_size=16):

  ix = torch.randint(len(data) - context_len-1, (batch_size,))
  x = torch.stack([data[i:i+context_len] for i in ix])       # (b, t)
  y = torch.stack([data[i+1:i+context_len+1] for i in ix])   # (b, t)
  return x, y

def stream(tokens_list):
  import time
  for x in tokens_list:
    time.sleep(0.2)
    print(tokenizer.idx_to_word[x.item()], end=" ")

data: tensor([178,  80,  30,   3, 189,  94, 105,  39,  70, 150,  80,  83,   3, 189,
        180,  59,  13,   3,  97, 160,  49, 179, 172, 105, 113,   3, 154,  46,
         78, 101, 182,  62, 167, 114, 195, 192,  20, 102,  50, 109, 183, 112,
        138,  96,  76, 167,  55,  93,  34,  31,  66, 144,  80,  23,  29, 106,
         70, 149,  73,  35,  24, 174, 135,  80,  53,   3, 108, 130, 162, 186,
        187,  45, 172, 162,  87,  26, 170,  13,  81,  69, 143,  22,  72,  80,
         88, 148,  67,  21,  69, 125, 132,  10, 117,  80, 122,  29,  71,  14,
        186, 181,  13, 131,  80,  12,  44,  65, 107,   8,  17,  69, 134, 165,
        109,  68,  93, 113,  54, 137,  13,  58,  10, 164,  80,  12, 116,  92,
         86,  64,  18,  90, 183, 191,   3, 105, 121, 156,  93,  19,  84,  19,
          7,  80,  12, 141, 161,  36, 186, 143,  22,  15, 109,  74,  93, 177,
         27, 173,  99,  95,  13, 119,  51,  80,  61, 115,  86,  75, 177, 169,
        133, 196, 123, 155, 153, 135,  80, 194, 113,  18, 

In [5]:
class RakGPT(nn.Module):

  def __init__(self, vocab_size, embed_dim, max_seq_len, num_heads, num_layers):
    super().__init__()
    self.vocab_size = vocab_size
    self.embed_dim = embed_dim
    self.max_seq_len = max_seq_len

    self.token_embedding = nn.Embedding(vocab_size, embed_dim)
    self.position_embedding = nn.Embedding(max_seq_len, embed_dim)

    self.blocks = nn.Sequential(
        *[TransformerBlock(embed_dim, max_seq_len, num_heads) for _ in range(num_layers)]
    )

    self.final_layer_norm = nn.LayerNorm(embed_dim)
    self.language_model_head = nn.Linear(embed_dim, vocab_size)


  def forward(self, idx, targets=None):
    """
    idx : (b, t)
    targets : (b, t) or None
    """

    batch_size, seq_len = idx.shape
    tok_emb = self.token_embedding(idx)
    pos = torch.arange(seq_len, device=idx.device)
    pos_emb = self.position_embedding(pos)

    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.final_layer_norm(x)
    logits = self.language_model_head(x)

    loss = None
    if targets is not None:
      loss = F.cross_entropy(
          logits.view(batch_size * seq_len, self.vocab_size),
          targets.view(batch_size * seq_len)
      )

    return logits, loss


  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
    """
    idx : (b, t_start)
    returns : (b, t_start + max_new_token)
    """
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.max_seq_len:]
      logits, _ = self(idx_cond)
      next_token_logits = logits[:, -1, :]
      probs = F.softmax(next_token_logits, dim=-1)
      next_idx = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, next_idx), dim=1)
    return idx


In [6]:
model = RakGPT(
    vocab_size=tokenizer.vocab_size,
    embed_dim=embedding_dim,
    max_seq_len=context_len,
    num_heads=n_heads,
    num_layers=n_layers
)

model

RakGPT(
  (token_embedding): Embedding(197, 64)
  (position_embedding): Embedding(6, 64)
  (blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-1): 2 x CausalSelfAttention(
            (query_proj): Linear(in_features=64, out_features=32, bias=False)
            (key_proj): Linear(in_features=64, out_features=32, bias=False)
            (value_proj): Linear(in_features=64, out_features=32, bias=False)
          )
        )
        (output_proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ffn): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
      )
      (attn_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (1): TransformerBl

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(epochs):
  xb, yb = get_batch()
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

  if step % 100 == 0:
    print(f"step: {step}, loss: {loss.item()}")

step: 0, loss: 5.493997097015381
step: 100, loss: 2.1457278728485107
step: 200, loss: 0.4629676640033722
step: 300, loss: 0.3644596040248871
step: 400, loss: 0.24798379838466644
step: 500, loss: 0.20829929411411285
step: 600, loss: 0.11154533177614212
step: 700, loss: 0.12561936676502228
step: 800, loss: 0.19004201889038086
step: 900, loss: 0.187111958861351
step: 1000, loss: 0.25666600465774536
step: 1100, loss: 0.22925575077533722
step: 1200, loss: 0.10401713103055954
step: 1300, loss: 0.1130811795592308
step: 1400, loss: 0.07346375286579132


In [31]:

prompt = "i am"

context_ids = tokenizer.encode(prompt)
context = torch.tensor([context_ids], dtype=torch.long)

out = model.generate(context, max_new_tokens=30)

stream(out[0])

i am particularly interested in explainable artificial intelligence<END> understanding why a model predicts something is as important as accuracy<END> i am researching supervised classification using retrieval based approaches<END> my goal is to 

In [33]:
import gradio as gr


def generate_text(prompt: str, max_new_tokens: int = 30):
    prompt = (prompt or "").strip()
    if not prompt:
        return ""


    context_ids = tokenizer.encode(prompt)
    context = torch.tensor([context_ids], dtype=torch.long)

    out = model.generate(context, max_new_tokens=max_new_tokens)


    decoded = tokenizer.decode(out[0].tolist(), stop_at_end=True)

    return decoded


with gr.Blocks() as demo:
    gr.Markdown("## RakGPT Demo (Word-level TinyGPT)")

    with gr.Row():
        prompt_in = gr.Textbox(label="Prompt", placeholder="Type something like: i am", value="i am")
        max_tokens = gr.Slider(
            minimum=1, maximum=100, value=30, step=1,
            label="Max new tokens"
        )

    run_btn = gr.Button("Generate")
    out_box = gr.Textbox(label="Output", lines=6)

    run_btn.click(
        fn=generate_text,
        inputs=[prompt_in, max_tokens],
        outputs=[out_box],
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24213657ceb9608211.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [38]:
import torch
import time

prompts = [
    "i am",
    "today i built",
    "my objective",
    "my passion",
]

MAX_NEW_TOKENS = 26

print("RakGPT (Small Language Model) Demo")

for i, prompt in enumerate(prompts, 1):
    print("=" * 60)
    print(f"Prompt {i}: {prompt}\n")
    print("Output:\n")

    context_ids = tokenizer.encode(prompt)
    context = torch.tensor([context_ids], dtype=torch.long)

    out = model.generate(context, max_new_tokens=MAX_NEW_TOKENS)

    stream(out[0])

    print("\n")
    time.sleep(0.7)

RakGPT (Small Language Model) Demo
Prompt 1: i am

Output:

i am extremely rewarding<END> i believe building images for retrieval based generation<END> i integrated scalable fastapi backends for production ready ai pipelines<END> i prefer building frontend applications using 

Prompt 2: today i built

Output:

today i built a vision language model completely from scratch<END> i implemented a vision transformer encoder and a llama style decoder<END> training the model on a small dataset helped 

Prompt 3: my objective

Output:

my objective is to build impactful artificial intelligence systems to multimodal models i enjoy exploring complex ai pipelines<END> my objective is to build impactful artificial intelligence systems thinking 

Prompt 4: my passion

Output:

my <UNK> building vidya mitra to provide ai powered learning for rural students<END> education accessibility through intelligent agents is a long term mission<END> i am exploring margin based 

